# 04 · ISPs by Region (Subdivision × ASN)

The finest non-city granularity: **state/province × ISP**. One row per country ×
subdivision × ASN per month.

## Why this matters

National ISP rankings (notebook 02) hide geographic variation. An ISP that ranks
10th nationally might be the best option in rural areas and absent from cities
entirely. This slice surfaces that variation.

The three-step cascade — month → country → subdivision → ISPs — lets you drill
from broad to narrow and focus on ISP competition in a specific region.

## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

try:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import seaborn as sns
    from IPython.display import clear_output, display
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 5)
except ImportError as e:
    print(f"Note: {e}")
    print("  Install with: uv add matplotlib seaborn ipywidgets")


In [2]:
# ── Country name lookup ──────────────────────────────────────────────────────
# countrylookup.py is a local helper (same directory as this notebook) that
# converts ISO 3166-1 alpha-2 codes to readable English country names.
# It tries pycountry → restcountries.com API → built-in fallback dict.
#
# If you move this notebook, keep countrylookup.py alongside it.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # ensure local module is found
from countrylookup import cc_name, cc_label

print(f"Country lookup ready — {cc_label('US')}, {cc_label('KR')}, {cc_label('XK')}")

[countrylookup] downloading country names from restcountries.com ...
Country lookup ready — United States (US), South Korea (KR), Kosovo (XK)


In [3]:
# ── Discover available months ─────────────────────────────────────────────────
#
# Fetch the M-Lab manifest to learn which months are available per slice.
# Each entry includes the public download URL and the local cache path.

MANIFEST_URL = "https://measurementlab.net/data/iqb/manifest.json"
resp = requests.get(MANIFEST_URL, timeout=30)
resp.raise_for_status()

records = []
for path, meta in resp.json()["files"].items():
    parts = path.split("/")
    if len(parts) >= 6 and parts[5] == "data.parquet":
        records.append({
            "start":      pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "slice":      parts[4],
            "url":        meta["url"],
            "cache_path": path,
        })

catalog = (
    pd.DataFrame(records)
    .sort_values(["slice", "start"])
    .reset_index(drop=True)
)

print(f"Catalog: {len(catalog)} entries, {catalog['slice'].nunique()} slices, "
      f"{catalog['start'].min().date()} → {catalog['start'].max().date()}")


Catalog: 2450 entries, 12 slices, 2009-01-01 → 2026-01-01


In [4]:
# ── Data loader ──────────────────────────────────────────────────────────────
#
# Downloads parquet files from the public M-Lab URLs in the manifest.
# Files are cached to ./cache/v1/... on first access and reused on subsequent
# runs (matching the path structure used by the iqb library's local cache).

from io import BytesIO

_mem_cache: dict = {}

def load_parquet(slice_name: str, start: str) -> pd.DataFrame:
    key = (slice_name, start)
    if key in _mem_cache:
        return _mem_cache[key]

    start_ts = pd.to_datetime(start)
    row = catalog[(catalog["slice"] == slice_name) & (catalog["start"] == start_ts)]
    if row.empty:
        available = catalog[catalog["slice"] == slice_name]["start"].dt.strftime("%Y-%m-%d").tolist()
        raise ValueError(f"No data for slice='{slice_name}', month='{start}'.\nAvailable: {available}")
    row = row.iloc[0]

    local_path = Path(row["cache_path"])
    if local_path.exists():
        df = pd.read_parquet(local_path)
    else:
        print(f"[download] {slice_name} / {start} …")
        r = requests.get(row["url"], timeout=60)
        r.raise_for_status()
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_bytes(r.content)
        df = pd.read_parquet(BytesIO(r.content))
        print(f"  ✓ saved to {local_path}  ({len(df):,} rows)")

    _mem_cache[key] = df
    return df


In [5]:
# ── ASN name lookup ──────────────────────────────────────────────────────────
#
# RIPE NCC publishes a plain-text file mapping every ASN to a human-readable
# organisation name. We cache it alongside the parquet files so subsequent
# sessions don't re-download it.
#
# Format:  {ASN}  {ORG-HANDLE} - {Full Name}, {Country}
# Source:  https://ftp.ripe.net/ripe/asnames/asn.txt

def load_asn_names() -> dict:
    cache_path = Path("cache/asn_names.txt")
    if cache_path.exists():
        text = cache_path.read_text(encoding="utf-8", errors="replace")
        print(f"[cache hit]  {cache_path}")
    else:
        print("[download]   RIPE ASN names ...")
        r = requests.get("https://ftp.ripe.net/ripe/asnames/asn.txt", timeout=30)
        r.raise_for_status()
        text = r.text
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(text, encoding="utf-8")
        print(f"  saved to {cache_path}")

    names = {}
    for line in text.splitlines():
        parts = line.strip().split(" ", 1)
        if len(parts) == 2:
            try:
                names[int(parts[0])] = parts[1]  # e.g. 'GOOGLE - Google LLC, US'
            except ValueError:
                pass
    return names

asn_names = load_asn_names()

# Helper: build a short display label for an ASN
def asn_label(asn: int, max_len: int = 45) -> str:
    name = asn_names.get(int(asn), "Unknown")
    # Strip trailing ', XX' country code for brevity
    if len(name) > 3 and name[-3] == ",":
        name = name[:-3].strip()
    label = f"AS{asn}: {name}"
    return label if len(label) <= max_len else label[:max_len - 1] + "…"

print(f"ASN names loaded: {len(asn_names):,} entries")
# Spot check
for test_asn in [7922, 15169, 3320, 1221]:
    print(f"  AS{test_asn}: {asn_names.get(test_asn, 'not found')}")

[cache hit]  cache/asn_names.txt
ASN names loaded: 121,463 entries
  AS7922: COMCAST-7922 - Comcast Cable Communications, LLC, US
  AS15169: GOOGLE - Google LLC, US
  AS3320: DTAG Deutsche Telekom AG, DE
  AS1221: ASN-TELSTRA - Telstra Limited, AU


## Interactive Explorer

1. **Pick a month** — data loads (or is served from disk cache)
2. **Pick a country** — subdivision list populates automatically
3. **Pick a subdivision** — ISPs in that region appear in the chart

In [6]:
sa_months = sorted(
    catalog[catalog["slice"] == "downloads_by_country_subdivision1_asn"]["start"]
    .dt.strftime("%Y-%m-%d").unique(), reverse=True,
)

# Metric options: display label → (column name, lower_is_better)
# Latency and loss: lower raw values are better, but IQB's polarity inversion
# means higher percentiles represent the better-performing connections.
METRICS = {
    "Download p50 (Mbit/s)":       ("download_p50",  False),
    "Upload p50 (Mbit/s)":         ("upload_p50",    False),
    "Latency p50 (ms)":            ("latency_p50",   True),
    "Packet Loss p50 (fraction)":  ("loss_p50",      True),
}

w_month   = widgets.Dropdown(options=sa_months, description="Month:",
                              layout=widgets.Layout(width="250px"))
w_country = widgets.Dropdown(options=[],         description="Country:",
                              layout=widgets.Layout(width="200px"))
w_sub     = widgets.Dropdown(options=[],         description="Region:",
                              layout=widgets.Layout(width="280px"))
w_min_s   = widgets.IntSlider(value=100, min=10, max=1000, step=10,
                               description="Min samples:", layout=widgets.Layout(width="380px"))
w_topn    = widgets.IntSlider(value=10, min=3, max=30, step=1,
                               description="Top N:", layout=widgets.Layout(width="380px"))
w_metric  = widgets.Dropdown(options=list(METRICS), description="Metric:",
                              layout=widgets.Layout(width="290px"))
out       = widgets.Output()

_sa: dict = {}
def _load_sa(month):
    if month in _sa: return _sa[month]
    df = load_parquet("downloads_by_country_subdivision1_asn", month)
    sc = next((c for c in df.columns if "subdivision" in c.lower()), "subdivision1")
    _sa[month] = (df, sc)
    return _sa[month]

def on_month(change):
    prev_country = w_country.value
    df, sc = _load_sa(change["new"])
    cc = sorted(df["country_code"].dropna().unique())
    w_country.options = [(cc_label(c), c) for c in cc]
    w_country.value = prev_country if prev_country in cc else ("US" if "US" in cc else cc[0])

def on_country(change):
    prev_sub = w_sub.value
    df, sc = _load_sa(w_month.value)
    subs = sorted(df[df["country_code"]==change["new"]][sc].dropna().unique())
    w_sub.options = subs
    # Restore previous subdivision if it exists in the new country
    w_sub.value = prev_sub if prev_sub in subs else (subs[0] if subs else None)

def draw(change=None):
    if not w_sub.value: return
    col, lower = METRICS[w_metric.value]
    df, sc = _load_sa(w_month.value)
    filt = df[(df["country_code"]==w_country.value) &
              (df[sc]==w_sub.value) &
              (df["sample_count"]>=w_min_s.value)]
    filt = filt.copy()
    filt["asn_label"] = filt["asn"].apply(asn_label)
    top = filt.nsmallest(w_topn.value,col) if lower else filt.nlargest(w_topn.value,col)
    top = top.sort_values(col, ascending=not lower)
    with out:
        clear_output(wait=True)
        if top.empty:
            print(f"No ASNs with >= {w_min_s.value} samples in '{w_sub.value}'. "
                  "Try lowering Min samples."); return
        fig, axes = plt.subplots(1,2,figsize=(14,max(4,w_topn.value*0.42)))
        axes[0].barh(top["asn_label"],top[col])
        axes[0].set_xlabel(w_metric.value)
        axes[0].set_title(f"{w_metric.value} {'(lower=better)' if lower else ''}")
        axes[1].barh(top["asn_label"],top["sample_count"],color="steelblue",alpha=0.7)
        axes[1].set_xlabel("Sample count")
        axes[1].set_title("Sample count")
        axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
        plt.suptitle(f"ISPs in '{w_sub.value}', {cc_name(w_country.value)} — {w_month.value}", fontsize=12)
        plt.tight_layout(); plt.show()

w_month.observe(on_month,"value"); w_country.observe(on_country,"value")
w_sub.observe(draw,"value"); w_min_s.observe(draw,"value")
w_topn.observe(draw,"value"); w_metric.observe(draw,"value")
display(widgets.VBox([widgets.HBox([w_month,w_country,w_sub]),
                      widgets.HBox([w_metric,w_min_s,w_topn]), out]))
on_month({"new": sa_months[0]})

[download] downloads_by_country_subdivision1_asn / 2025-12-01 …
  ✓ saved to cache/v1/20251201T000000Z/20260101T000000Z/downloads_by_country_subdivision1_asn/data.parquet  (47,066 rows)
